# XClinVision: Uncertainty Quantification Analysis
**Day 4: Uncertainty & Calibration**

This notebook demonstrates:
- MC Dropout uncertainty estimation
- Temperature scaling calibration
- Expected Calibration Error (ECE) analysis
- Confidence distribution visualization

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from xclinvision.evaluator import CalibrationAnalyzer, TemperatureScaler
from xclinvision.inference import InferencePipeline

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

## 2. Generate Mock Predictions for Demo
*(In practice, use actual model predictions on test set)*

In [ ]:
# Simulate predictions and ground truth
np.random.seed(42)
n_samples = 1000

# Well-calibrated model predictions
y_true = np.random.randint(0, 3, n_samples)
y_pred_probs = np.random.dirichlet([2, 2, 2], n_samples)

# Add some correlation with true labels for realism
for i in range(n_samples):
    y_pred_probs[i, y_true[i]] += 0.5
    y_pred_probs[i] /= y_pred_probs[i].sum()

y_pred = np.argmax(y_pred_probs, axis=1)
confidence = np.max(y_pred_probs, axis=1)

print(f"Samples: {n_samples}")
print(f"Accuracy: {(y_pred == y_true).mean():.3f}")
print(f"Mean confidence: {confidence.mean():.3f}")

## 3. Expected Calibration Error (ECE)

In [ ]:
def compute_ece(y_true, y_pred_probs, n_bins=15):
    """Compute Expected Calibration Error."""
    
    confidences = np.max(y_pred_probs, axis=1)
    predictions = np.argmax(y_pred_probs, axis=1)
    accuracies = (predictions == y_true).astype(float)
    
    # Create bins
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0.0
    bin_accs = []
    bin_confs = []
    bin_counts = []
    
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            accuracy_in_bin = accuracies[in_bin].mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
            bin_accs.append(accuracy_in_bin)
            bin_confs.append(avg_confidence_in_bin)
            bin_counts.append(in_bin.sum())
        else:
            bin_accs.append(0)
            bin_confs.append(0)
            bin_counts.append(0)
    
    return ece, bin_accs, bin_confs, bin_counts, bin_boundaries

# Compute ECE
ece, bin_accs, bin_confs, bin_counts, bin_boundaries = compute_ece(y_true, y_pred_probs)
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 4. Calibration Curve (Reliability Diagram)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram
ax = axes[0]
bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2

# Perfect calibration line
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")

# Actual calibration
ax.bar(bin_centers, bin_accs, width=0.06, alpha=0.6, edgecolor="black", label="Model Accuracy")

# Gap bars
for i, (center, acc, conf) in enumerate(zip(bin_centers, bin_accs, bin_confs)):
    if bin_counts[i] > 0:
        ax.plot([center, center], [acc, conf], "r-", linewidth=2)

ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.set_title(f"Reliability Diagram (ECE={ece:.3f})")
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)

# Confidence histogram
ax2 = axes[1]
ax2.hist(confidence, bins=20, edgecolor="black", alpha=0.7)
ax2.set_xlabel("Confidence")
ax2.set_ylabel("Count")
ax2.set_title("Confidence Distribution")
ax2.axvline(confidence.mean(), color="r", linestyle="--", label=f"Mean={confidence.mean():.3f}")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. MC Dropout Uncertainty Demo

In [ ]:
def mc_dropout_uncertainty_demo(n_samples=10, n_classes=3):
    """Simulate MC Dropout uncertainty estimation."""
    
    # Simulate multiple forward passes with dropout
    np.random.seed(42)
    n_forward_passes = 50
    
    # Generate varying predictions (simulating dropout uncertainty)
    predictions = []
    for _ in range(n_forward_passes):
        # Add noise to simulate dropout randomness
        noise = np.random.randn(n_samples, n_classes) * 0.3
        logits = np.array([[2.0, 1.0, 0.5]] * n_samples) + noise
        probs = F.softmax(torch.tensor(logits), dim=1).numpy()
        predictions.append(probs)
    
    predictions = np.array(predictions)  # (n_forward, n_samples, n_classes)
    
    # Compute uncertainty metrics
    mean_probs = predictions.mean(axis=0)
    epistemic_unc = predictions.var(axis=0).mean(axis=1)  # Variance across forward passes
    predictive_entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
    
    return mean_probs, epistemic_unc, predictive_entropy

mean_probs, epistemic_unc, predictive_entropy = mc_dropout_uncertainty_demo()

# Visualize uncertainty distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(epistemic_unc, bins=20, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Epistemic Uncertainty (Variance)")
axes[0].set_ylabel("Count")
axes[0].set_title("Epistemic Uncertainty Distribution")

axes[1].hist(predictive_entropy, bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Predictive Entropy")
axes[1].set_ylabel("Count")
axes[1].set_title("Predictive Entropy Distribution")

plt.tight_layout()
plt.show()

print(f"Mean epistemic uncertainty: {epistemic_unc.mean():.4f}")
print(f"Mean predictive entropy: {predictive_entropy.mean():.4f}")